# LNG Fleet Portfolio Optimiser
## C1 Fleet (6 vessels) + C2 Acquisition Targets (W1, W2)

**Project:** Stochastic portfolio optimisation for an 8-vessel LNG carrier fleet  
**Version:** v7 (regime-switching Monte Carlo)  
**Date:** March 2026  

---

### What this notebook does

This notebook implements a full stochastic portfolio optimisation model for an LNG shipping fleet.  
It covers six modelling layers:

1. **Route economics** — deterministic per-cargo margin for every feasible ship-route combination  
2. **Ship assignment** — bucket each vessel as Committed (DES) or Swing (FOB) with hedge ratios  
3. **Price simulation** — correlated GBM for Brent, HH, TTF, JKM + OU for LNG freight  
4. **Regime switching** — two-state Markov model (normal / stress) with separate correlation matrices  
5. **Two-stage optimisation** — fix committed ships at Stage 1; re-route FOB swing ships at Stage 2  
6. **Acquisition analysis** — marginal portfolio value of adding C2 vessels W1 and W2  

### Fleet

| # | Vessel | m³ | Type | Origin | Hedge |
|---|--------|----|------|--------|-------|
| 1 | GasLog Italy | 174k | DES | US (HH) | 85% |
| 2 | Maran Gas Kalymnos | 174k | FOB swing | Nigeria (Brent) | 45% |
| 3 | Maran Gas Kastelorizo | 174k | DES | US (HH) | 80% |
| 4 | Maran Gas Achilles | 174k | DES | Australia (Brent) | 90% |
| 5 | Maran Gas Delphi | 160k | FOB swing | Nigeria (Brent) | 40% |
| 6 | Seapeak Galicia | 140k | FOB swing | US (HH) | 50% |
| 7 | W1 (C2) | 156k | FOB swing | Atlantic (Mixed) | — |
| 8 | W2 (C2) | 156k | FOB swing | Atlantic/AUS (Mixed) | — |


## 0 — Imports & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import norm
from scipy.linalg import cholesky
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.facecolor': '#0d1b2a',
    'axes.facecolor':   '#0d1b2a',
    'axes.edgecolor':   '#1e3a5f',
    'text.color':       '#b0c4d8',
    'axes.labelcolor':  '#b0c4d8',
    'xtick.color':      '#7e99b4',
    'ytick.color':      '#7e99b4',
    'grid.color':       '#1e3a5f',
    'grid.alpha':       0.5,
    'axes.titlecolor':  '#e0eaf5',
    'figure.titlesize': 13,
})
BLUE, TEAL, AMBER, RED, GREEN = '#42a5f5','#4db6ac','#c59a2e','#ef5350','#66bb6a'

np.random.seed(42)
print("Imports OK")


## 1 — Market parameters & route economics

### 1.1 Current market benchmarks (March 2026)

All prices in USD/MMBtu unless stated.


In [ ]:
# ── Spot benchmarks ─────────────────────────────────────────────────
BRENT   = 82.00   # USD/bbl
HH      = 2.40    # USD/MMBtu
TTF     = 11.50   # USD/MMBtu
JKM     = 13.20   # USD/MMBtu

# ── FOB supply costs ─────────────────────────────────────────────────
FOB_US  = HH + 3.00          # Sabine Pass / Corpus Christi liquefaction
FOB_NGA = 0.08 * BRENT       # Nigeria DES — Brent-linked
FOB_AUS = 0.12 * BRENT       # Australia — Brent-linked higher slope

print(f"FOB US:  ${FOB_US:.2f}/MMBtu  (HH + $3.00 liq fee)")
print(f"FOB NGA: ${FOB_NGA:.2f}/MMBtu  (0.08 × Brent)")
print(f"FOB AUS: ${FOB_AUS:.2f}/MMBtu  (0.12 × Brent)")
print(f"JKM-TTF spread: ${JKM-TTF:.2f}/MMBtu")


### 1.2 Route matrix

**Margin formula (v7):**
```
Margin = DES_price − FOB_cost − Freight − Boil-off(DES price) − Port − Canal − E[Demurrage]
```

**v7 fixes vs v6:**
- Boil-off valued at DES price (not flat $12/MMBtu)
- Vessel-specific freight (140k pays ~20% more per MMBtu than 174k)
- Full round-trip cycle: laden + discharge + ballast + waiting
- Demurrage included as expected value per port call


In [ ]:
# ── Route definitions ────────────────────────────────────────────────
# (origin, destination, FOB, DES, freight_174k, rt_days, laden_days, port, canal, demurrage)
routes_raw = [
    ('US',  'EU',   FOB_US,  TTF,  0.60, 28, 14, 0.20, 0.05, 0.08),
    ('US',  'Asia', FOB_US,  JKM,  0.95, 52, 26, 0.20, 0.15, 0.12),
    ('NGA', 'EU',   FOB_NGA, TTF,  0.65, 18, 10, 0.20, 0.00, 0.07),
    ('NGA', 'Asia', FOB_NGA, JKM,  0.85, 38, 20, 0.20, 0.12, 0.10),
    ('AUS', 'Asia', FOB_AUS, JKM,  0.55, 19, 10, 0.20, 0.00, 0.06),
    ('AUS', 'EU',   FOB_AUS, TTF,  1.10, 60, 30, 0.20, 0.10, 0.15),
]

CARGO_174K = 174_000 * 21.17 / 1e6   # 174k m³ → MMBtu (millions)
BOR = 0.00085  # boil-off rate per day

def calc_margin(fob, des, freight, laden_days, port, canal, demurrage, bor=BOR):
    boiloff = bor * laden_days * des   # v7: at DES price
    return des - fob - freight - boiloff - port - canal - demurrage

cols = ['Origin','Dest','FOB','DES','Freight','RT_days','Laden_days',
        'Port','Canal','Demurrage','Margin','Margin_per_day']
rows = []
for r in routes_raw:
    org, dst, fob, des, fr, rt, ld, port, canal, dem = r
    margin = calc_margin(fob, des, fr, ld, port, canal, dem)
    rows.append([org, dst, fob, des, fr, rt, ld, port, canal, dem,
                 margin, margin / rt])

df_routes = pd.DataFrame(rows, columns=cols).sort_values('Margin_per_day', ascending=False)
print(df_routes[['Origin','Dest','FOB','DES','Margin','RT_days','Margin_per_day']].to_string(index=False, float_format='${:.3f}'.format))


## 2 — Ship assignment & hedge ratios

In [ ]:
# ── Fleet definition ─────────────────────────────────────────────────
ships = pd.DataFrame([
    # id, name,            cap_m3, mmBtu_M, supply, dest,   contract, hedge, open_pl, sig
    (1, 'GasLog Italy',      174000, 3.689, 'HH',    'TTF', 'DES', 0.85, 2.53, 3.82),
    (2, 'Kalymnos',          174000, 3.689, 'Brent', 'JKM', 'FOB', 0.45, 9.31, 4.21),
    (3, 'Kastelorizo',       174000, 3.689, 'Brent', 'TTF', 'DES', 0.80, 2.95, 2.94),
    (4, 'Achilles',          174000, 3.689, 'Brent', 'JKM', 'DES', 0.90, 0.99, 1.44),
    (5, 'Delphi',            160000, 3.392, 'Brent', 'TTF', 'FOB', 0.40, 8.16, 4.18),
    (6, 'Galicia',           140000, 2.968, 'HH',    'TTF', 'FOB', 0.50, 7.86, 3.74),
    (7, 'W1',                156000, 3.307, 'Mixed', 'TTF', 'FOB', 0.40, 6.80, 3.50),
    (8, 'W2',                156000, 3.307, 'Mixed', 'JKM', 'FOB', 0.35, 4.90, 3.20),
], columns=['id','name','cap_m3','cargo_MMBtu','supply_bm','dest_bm',
            'contract','hedge_ratio','open_pl_M','sigma_M'])

# Swing ships (FOB = flexible destination)
swing = ships[ships.contract == 'FOB'].copy()
committed = ships[ships.contract == 'DES'].copy()

print(f"Fleet: {len(ships)} ships | {len(committed)} committed DES | {len(swing)} FOB swing")
print(f"Total locked P&L: ${ships.open_pl_M.sum() * 0.0:.1f}M (open exposure shown below)")
print()
print(ships[['id','name','contract','hedge_ratio','open_pl_M','sigma_M']].to_string(index=False))


## 3 — Stochastic simulation

### 3.1 Price model parameters

- **Brent, HH, TTF, JKM**: Geometric Brownian Motion (GBM) — log-normal, no mean reversion
- **LNG Freight**: Ornstein-Uhlenbeck (OU) — mean-reverting, half-life ≈ 58 days
- **Simulation**: 5,000 paths × 90 days, daily steps
- **Correlation**: Cholesky decomposition of regime-dependent correlation matrix


In [ ]:
# ── Simulation parameters ────────────────────────────────────────────
N_PATHS  = 5000
N_DAYS   = 90
DT       = 1/365

# GBM volatilities (annual)
vols = {'Brent': 0.28, 'HH': 0.45, 'TTF': 0.35, 'JKM': 0.30}

# Freight OU parameters
FREIGHT_KAPPA = 3.0   # mean-reversion speed (half-life = ln2/kappa ≈ 84d)
FREIGHT_THETA = 1.0   # long-run mean (index)
FREIGHT_SIGMA = 0.35  # vol

# ── Correlation matrices ──────────────────────────────────────────────
# Order: [Brent, HH, TTF, JKM]
CORR_NORMAL = np.array([
    [1.00, 0.15, 0.40, 0.35],
    [0.15, 1.00, 0.25, 0.20],
    [0.40, 0.25, 1.00, 0.75],
    [0.35, 0.20, 0.75, 1.00],
])

CORR_STRESS = np.array([
    [1.00, 0.05, 0.65, 0.55],
    [0.05, 1.00, 0.10, 0.08],
    [0.65, 0.10, 1.00, 0.40],
    [0.55, 0.08, 0.40, 1.00],
])

# Cholesky factors
L_NORMAL = cholesky(CORR_NORMAL, lower=True)
L_STRESS = cholesky(CORR_STRESS, lower=True)

print("Correlation matrices validated (positive definite):")
print(f"  Normal regime eigenvalues: {np.linalg.eigvals(CORR_NORMAL).round(3)}")
print(f"  Stress regime eigenvalues: {np.linalg.eigvals(CORR_STRESS).round(3)}")


### 3.2 Regime-switching Markov chain

Two regimes transition monthly via a Markov chain:

| Transition | Probability | Meaning |
|---|---|---|
| Normal → Stress | 3%/month | Supply crisis onset |
| Stress → Normal | 20%/month | Market normalisation |

**Stationary probabilities:** π(normal) = 87%, π(stress) = 13%  

**Key insight:** TTF-JKM correlation *drops* in stress (0.75 → 0.40). European and Asian gas markets decouple during supply crises — exactly when swing ship optionality is most valuable.


In [ ]:
# ── Regime-switching parameters ──────────────────────────────────────
P_NORMAL_TO_STRESS = 0.03   # per month
P_STRESS_TO_NORMAL = 0.20   # per month
REGIME_NORMAL, REGIME_STRESS = 0, 1

# Stationary distribution
pi_stress = P_NORMAL_TO_STRESS / (P_NORMAL_TO_STRESS + P_STRESS_TO_NORMAL)
pi_normal = 1 - pi_stress
print(f"Stationary: π(normal)={pi_normal:.1%}  π(stress)={pi_stress:.1%}")
print(f"Avg spell: normal={1/P_NORMAL_TO_STRESS:.0f}mo  stress={1/P_STRESS_TO_NORMAL:.0f}mo")

def simulate_regimes(n_paths, n_days):
    """Simulate regime sequence for all paths. Returns (n_paths, n_days) int array."""
    regimes = np.zeros((n_paths, n_days), dtype=int)
    # Initial regime drawn from stationary distribution
    regimes[:, 0] = (np.random.rand(n_paths) < pi_stress).astype(int)
    # Monthly transitions (every ~21 trading days)
    for t in range(1, n_days):
        if t % 21 == 0:
            in_normal = regimes[:, t-1] == REGIME_NORMAL
            in_stress = regimes[:, t-1] == REGIME_STRESS
            flip_to_stress = in_normal & (np.random.rand(n_paths) < P_NORMAL_TO_STRESS)
            flip_to_normal = in_stress & (np.random.rand(n_paths) < P_STRESS_TO_NORMAL)
            regimes[:, t] = regimes[:, t-1].copy()
            regimes[flip_to_stress, t] = REGIME_STRESS
            regimes[flip_to_normal, t] = REGIME_NORMAL
        else:
            regimes[:, t] = regimes[:, t-1]
    return regimes

# Test
test_reg = simulate_regimes(10000, 90)
stress_pct = test_reg.mean()
print(f"Simulated stress fraction: {stress_pct:.1%} (target ~13%)")


### 3.3 Simulate correlated price paths

In [ ]:
def simulate_prices(n_paths=N_PATHS, n_days=N_DAYS):
    """
    Simulate correlated GBM paths for Brent, HH, TTF, JKM
    plus OU freight, with regime-switching correlation.
    
    Returns dict of (n_paths, n_days+1) arrays.
    """
    S0 = np.array([BRENT, HH, TTF, JKM])
    sig = np.array([vols['Brent'], vols['HH'], vols['TTF'], vols['JKM']])
    drift = -0.5 * sig**2 * DT  # Ito correction

    # Storage: [n_paths, n_days+1]
    prices = np.zeros((4, n_paths, n_days + 1))
    prices[:, :, 0] = S0[:, np.newaxis]

    freight = np.zeros((n_paths, n_days + 1))
    freight[:, 0] = FREIGHT_THETA

    # Regimes
    regimes = simulate_regimes(n_paths, n_days)

    for t in range(n_days):
        # Draw correlated normals — split by regime
        Z_raw = np.random.randn(4, n_paths)
        in_stress = regimes[:, t] == REGIME_STRESS

        # Normal regime paths
        if np.any(~in_stress):
            idx = ~in_stress
            Z_corr = L_NORMAL @ Z_raw[:, idx]
            prices[:, idx, t+1] = prices[:, idx, t] * np.exp(
                drift[:, np.newaxis] + sig[:, np.newaxis] * np.sqrt(DT) * Z_corr
            )

        # Stress regime paths
        if np.any(in_stress):
            idx = in_stress
            Z_corr = L_STRESS @ Z_raw[:, idx]
            prices[:, idx, t+1] = prices[:, idx, t] * np.exp(
                drift[:, np.newaxis] + sig[:, np.newaxis] * np.sqrt(DT) * Z_corr
            )

        # OU freight
        Z_f = np.random.randn(n_paths)
        freight[:, t+1] = (freight[:, t]
                           + FREIGHT_KAPPA * (FREIGHT_THETA - freight[:, t]) * DT
                           + FREIGHT_SIGMA * np.sqrt(DT) * Z_f)

    return {
        'Brent':   prices[0],
        'HH':      prices[1],
        'TTF':     prices[2],
        'JKM':     prices[3],
        'Freight': freight,
        'regimes': regimes,
    }

print("Running simulation...")
sim = simulate_prices()
print(f"Shapes — Brent: {sim['Brent'].shape}, Freight: {sim['Freight'].shape}")
print(f"\nEnd-of-horizon statistics (day 90):")
for k in ['Brent','HH','TTF','JKM']:
    vals = sim[k][:, -1]
    print(f"  {k:6s}: mean=${vals.mean():.2f}  std=${vals.std():.2f}  "
          f"p5=${np.percentile(vals,5):.2f}  p95=${np.percentile(vals,95):.2f}")


### 3.4 Visualise simulated paths

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Simulated Price Paths — 200 representative paths (5,000 total)', 
             fontsize=12, color='#e0eaf5', y=1.01)

plot_vars = [
    ('TTF',     TTF,   BLUE,  'TTF (EU gas)'),
    ('JKM',     JKM,   TEAL,  'JKM (Asia LNG)'),
    ('Brent',   BRENT, AMBER, 'Brent crude'),
    ('HH',      HH,    GREEN, 'Henry Hub'),
    ('Freight', 1.0,   RED,   'LNG Freight (OU index)'),
]

t = np.arange(N_DAYS + 1)
sample = np.random.choice(N_PATHS, 200, replace=False)

for ax, (var, s0, col, title) in zip(axes.flat, plot_vars):
    paths = sim[var][sample, :]
    ax.plot(t, paths.T, color=col, alpha=0.04, lw=0.6)
    ax.plot(t, np.median(sim[var], axis=0), color=col, lw=2, label='Median')
    ax.plot(t, np.percentile(sim[var], 5,  axis=0), color='white', lw=1, ls='--', alpha=0.5)
    ax.plot(t, np.percentile(sim[var], 95, axis=0), color='white', lw=1, ls='--', alpha=0.5)
    ax.axhline(s0, color='white', lw=0.8, ls=':', alpha=0.4)
    ax.set_title(title, fontsize=10, color='#e0eaf5')
    ax.set_xlabel('Days', fontsize=9)
    ax.grid(True, alpha=0.3)

# Regime frequency
ax = axes.flat[-1]
stress_by_day = (sim['regimes'] == REGIME_STRESS).mean(axis=0)
ax.fill_between(np.arange(N_DAYS), stress_by_day * 100, 
                color=RED, alpha=0.6, label='Stress %')
ax.axhline(pi_stress * 100, color='white', lw=1, ls='--', alpha=0.5, label='Stationary')
ax.set_title('Stress regime frequency (%)', fontsize=10, color='#e0eaf5')
ax.set_xlabel('Days', fontsize=9)
ax.set_ylim(0, 40)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('price_paths.png', dpi=120, bbox_inches='tight', facecolor='#0d1b2a')
plt.show()
print("Saved price_paths.png")


## 4 — Portfolio P&L calculation

### 4.1 Per-ship P&L model

Each ship's P&L has two components:
- **Locked**: hedged portion at locked-in prices (deterministic)
- **Open**: unhedged portion exposed to spread movements

Open P&L scales with spread volatility relative to base case:
```
spread_vol = sqrt(σ_supply² + σ_dest² − 2ρ·σ_supply·σ_dest)
adj_factor = spread_vol_simulated / spread_vol_base
open_PL_adj = open_PL_base × adj_factor
```


In [ ]:
# ── Locked P&L per ship (deterministic) ──────────────────────────────
LOCKED_PL = {
    1: 14.34, 2: 7.62,  3: 11.81, 4: 8.87,
    5: 5.44,  6: 7.86,  7: 8.40,  8: 6.00,
}

# ── Compute portfolio P&L for each scenario ───────────────────────────
def compute_portfolio_pl(sim, ships_df, fleet='8ship'):
    """
    Compute scenario-level portfolio P&L.
    
    Strategy:
      - Committed ships: locked_PL + open_PL scaled by spread vol ratio
      - Swing ships (Stage 1): same as committed  
      - Swing ships (Stage 2): re-route to best destination per scenario
    
    Returns array of shape (N_PATHS,)
    """
    n = sim['Brent'].shape[0]
    
    # End-of-horizon prices
    brent = sim['Brent'][:, -1]
    hh    = sim['HH'][:, -1]
    ttf   = sim['TTF'][:, -1]
    jkm   = sim['JKM'][:, -1]
    
    # Regime at horizon
    stress = (sim['regimes'][:, -1] == REGIME_STRESS)

    if fleet == '6ship':
        ship_list = ships_df[ships_df.id <= 6]
    else:
        ship_list = ships_df

    total_pl = np.zeros(n)

    for _, s in ship_list.iterrows():
        locked = LOCKED_PL[s.id]

        # Spread vol ratio: actual vs base
        sig_supply = 0.45 if s.supply_bm == 'HH' else 0.28 if s.supply_bm == 'Brent' else 0.36
        sig_dest   = 0.35 if s.dest_bm == 'TTF' else 0.30

        # Base spread vol (normal regime correlations)
        rho_base = (CORR_NORMAL[1, 2] if s.supply_bm == 'HH' and s.dest_bm == 'TTF'
                    else CORR_NORMAL[1, 3] if s.supply_bm == 'HH' and s.dest_bm == 'JKM'
                    else CORR_NORMAL[0, 2] if s.dest_bm == 'TTF'
                    else CORR_NORMAL[0, 3])
        sv_base = np.sqrt(sig_supply**2 + sig_dest**2 - 2*rho_base*sig_supply*sig_dest)

        # Scenario spread vol (regime-dependent)
        rho_n = rho_base
        rho_s = (CORR_STRESS[1, 2] if s.supply_bm == 'HH' and s.dest_bm == 'TTF'
                 else CORR_STRESS[1, 3] if s.supply_bm == 'HH' and s.dest_bm == 'JKM'
                 else CORR_STRESS[0, 2] if s.dest_bm == 'TTF'
                 else CORR_STRESS[0, 3])
        sv_scen = np.where(stress,
                           np.sqrt(sig_supply**2 + sig_dest**2 - 2*rho_s*sig_supply*sig_dest),
                           np.sqrt(sig_supply**2 + sig_dest**2 - 2*rho_n*sig_supply*sig_dest))

        adj_factor = sv_scen / sv_base
        open_adj = s.open_pl_M * adj_factor

        if s.contract == 'FOB':
            # Stage 2: swing ships pick best destination margin
            fob = hh + 3.00 if s.supply_bm == 'HH' else 0.08 * brent if s.supply_bm == 'Brent' else 0.09 * brent
            margin_eu   = ttf - fob - 0.60 - 0.003 * 14 * ttf - 0.25
            margin_asia = jkm - fob - 0.85 - 0.003 * 20 * jkm - 0.32
            best_margin = np.maximum(margin_eu, margin_asia)
            # Scale open PL by best vs base margin ratio
            base_margin = s.open_pl_M / (s.cargo_MMBtu * (1 - s.hedge_ratio))
            scale = np.clip(best_margin / (base_margin + 1e-6), 0.5, 2.0)
            open_adj = s.open_pl_M * scale * adj_factor

        total_pl += locked + open_adj

    return total_pl

pl_6  = compute_portfolio_pl(sim, ships, fleet='6ship')
pl_8  = compute_portfolio_pl(sim, ships, fleet='8ship')

print("=== 6-ship C1 fleet ===")
print(f"  E[P&L]:  ${pl_6.mean():.2f}M")
print(f"  Std Dev: ${pl_6.std():.2f}M")
print(f"  P5:      ${np.percentile(pl_6, 5):.2f}M")
print(f"  CVaR:    ${pl_6[pl_6 <= np.percentile(pl_6, 5)].mean():.2f}M")
print()
print("=== 8-ship fleet (C1 + C2) ===")
print(f"  E[P&L]:  ${pl_8.mean():.2f}M")
print(f"  Std Dev: ${pl_8.std():.2f}M")
print(f"  P5:      ${np.percentile(pl_8, 5):.2f}M")
print(f"  CVaR:    ${pl_8[pl_8 <= np.percentile(pl_8, 5)].mean():.2f}M")


### 4.2 P&L distribution — 6-ship vs 8-ship

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Portfolio P&L Distribution — 5,000 Monte Carlo Scenarios', 
             fontsize=12, color='#e0eaf5')

for ax, (pl, label, col, fleet) in zip(axes, [
    (pl_6, 'C1 6-ship', BLUE, '6ship'),
    (pl_8, 'C1+C2 8-ship', TEAL, '8ship'),
]):
    ax.hist(pl, bins=80, color=col, alpha=0.7, edgecolor='none', density=True)
    
    p5  = np.percentile(pl, 5)
    cvar = pl[pl <= p5].mean()
    mean = pl.mean()
    
    ax.axvline(mean, color='white',  lw=1.5, ls='-',  label=f'E[P&L] ${mean:.1f}M')
    ax.axvline(p5,   color=RED,      lw=1.5, ls='--', label=f'P5 ${p5:.1f}M')
    ax.axvline(cvar, color=AMBER,    lw=1.5, ls=':',  label=f'CVaR ${cvar:.1f}M')
    
    # Shade tail
    tail = pl[pl <= p5]
    ax.hist(tail, bins=30, color=RED, alpha=0.5, density=True, edgecolor='none')
    
    ax.set_title(f'{label}', fontsize=11, color='#e0eaf5')
    ax.set_xlabel('P&L ($M)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pl_distribution.png', dpi=120, bbox_inches='tight', facecolor='#0d1b2a')
plt.show()


## 5 — Real options: Margrabe exchange option

Each FOB swing ship holds an option to exchange one destination for another.  
This is priced using the **Margrabe formula** for exchange options:

$$V = Q \cdot [M_1 \cdot N(d_1) - M_2 \cdot N(d_2)]$$

$$d_1 = \frac{\ln(M_1/M_2) + \frac{1}{2}\sigma^2 T}{\sigma \sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

$$\sigma^2 = \sigma_1^2 + \sigma_2^2 - 2\rho \cdot \sigma_1 \cdot \sigma_2 \quad (\text{spread vol})$$

where $M_1, M_2$ are the two destination margins and $\rho$ is the TTF-JKM correlation.


In [ ]:
def margrabe_option(M1, M2, sigma1, sigma2, rho, T, cargo_MMBtu):
    """
    Margrabe exchange option value.
    Option to receive max(M1, M2) instead of M1.
    
    Args:
        M1, M2: $/MMBtu margins for default and alternative destinations
        sigma1, sigma2: annual vols of the two margins
        rho: correlation between the two margins
        T: time horizon (years)
        cargo_MMBtu: cargo size in millions of MMBtu
    
    Returns:
        option_value ($M)
    """
    sigma = np.sqrt(sigma1**2 + sigma2**2 - 2*rho*sigma1*sigma2)
    if sigma < 1e-8 or M1 <= 0 or M2 <= 0:
        return max(M1 - M2, 0) * cargo_MMBtu
    d1 = (np.log(M1/M2) + 0.5*sigma**2*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    value = cargo_MMBtu * (M1*norm.cdf(d1) - M2*norm.cdf(d2))
    return max(value, 0)

# ── Swing ship option values ──────────────────────────────────────────
T = 45/365  # decision horizon ~45 days

# TTF-JKM spread vol
SIG_TTF = 0.35; SIG_JKM = 0.30

swing_options = [
    # name,    M1,   M2,   cargo,  T,    note
    ('Kalymnos (2)', 5.00, 4.32, 3.689, T,  'NGA→Asia dflt; option to EU'),
    ('Delphi (5)',   3.97, 4.50, 3.392, T,  'NGA→EU dflt; option to Asia'),
    ('Galicia (6)',  4.55, 3.80, 2.968, T,  'US→EU dflt; option to Asia'),
    ('W1',           4.50, 3.90, 3.307, T,  'Atlantic→EU dflt; option to Asia'),
    ('W2',           3.50, 2.80, 3.307, T,  'AUS→Asia dflt; option to EU'),
]

print(f"{'Ship':<20} {'M1':>6} {'M2':>6} {'σ_spread':>10} {'T(yr)':>8} {'Option $M':>10}  Note")
print('-' * 85)
total_option = 0
for name, m1, m2, cargo, t, note in swing_options:
    # Normal regime
    rho_n = CORR_NORMAL[2, 3]
    v_n = margrabe_option(m1, m2, SIG_TTF, SIG_JKM, rho_n, t, cargo)
    # Stress regime (TTF-JKM drops to 0.40 → option value rises)
    rho_s = CORR_STRESS[2, 3]
    v_s = margrabe_option(m1, m2, SIG_TTF, SIG_JKM, rho_s, t, cargo)
    sv = np.sqrt(SIG_TTF**2 + SIG_JKM**2 - 2*rho_n*SIG_TTF*SIG_JKM)
    print(f"{name:<20} {m1:>6.2f} {m2:>6.2f} {sv:>10.3f} {t:>8.3f} {v_n:>10.3f}  {note}")
    print(f"{'':20} {'stress uplift:':>37} +{v_s-v_n:.3f}M ({(v_s/v_n-1)*100:.0f}%)")
    total_option += v_n

print(f"\nTotal option value (5 swing ships, normal regime): ${total_option:.2f}M")
print(f"In stress regime, TTF-JKM ρ: {rho_n:.2f} → {rho_s:.2f}, spread vol widens → option values ↑~55%")


## 6 — Hedge ratio sweep

Optimal hedge ratio trades off E[P&L] against CVaR.  
Higher hedge → lower E[P&L] (less optionality) but better CVaR (less tail exposure).  
Operating point: committed ships 80-90%, swing ships 35-50%.


In [ ]:
def portfolio_pl_at_hedge(sim, ships_df, committed_hr, swing_hr):
    """Simplified P&L at given hedge ratios."""
    n = sim['Brent'].shape[0]
    ttf = sim['TTF'][:, -1]
    jkm = sim['JKM'][:, -1]
    brent = sim['Brent'][:, -1]
    hh = sim['HH'][:, -1]
    
    total = np.zeros(n)
    for _, s in ships_df[ships_df.id <= 6].iterrows():
        hr = committed_hr if s.contract == 'DES' else swing_hr
        locked_adj = LOCKED_PL[s.id] * (hr / s.hedge_ratio)
        open_adj   = s.open_pl_M * ((1 - hr) / (1 - s.hedge_ratio))
        total += locked_adj + open_adj
    return total

# Sweep
committed_range = np.arange(0.60, 1.01, 0.05)
swing_range     = np.arange(0.20, 0.71, 0.05)
C_HR, S_HR = np.meshgrid(committed_range, swing_range)

EPL  = np.zeros_like(C_HR)
CVAR = np.zeros_like(C_HR)

for i in range(C_HR.shape[0]):
    for j in range(C_HR.shape[1]):
        pl = portfolio_pl_at_hedge(sim, ships, C_HR[i,j], S_HR[i,j])
        EPL[i,j]  = pl.mean()
        p5 = np.percentile(pl, 5)
        CVAR[i,j] = pl[pl <= p5].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Hedge Ratio Sweep — E[P&L] and CVaR ($M)', fontsize=12, color='#e0eaf5')

for ax, (data, title, cmap) in zip(axes, [
    (EPL,  'E[P&L]', 'Blues'),
    (CVAR, 'CVaR',   'Greens'),
]):
    im = ax.contourf(committed_range, swing_range, data, levels=20, cmap=cmap, alpha=0.85)
    ax.contour(committed_range,  swing_range, data, levels=10, colors='white', alpha=0.2, linewidths=0.5)
    plt.colorbar(im, ax=ax, label='$M')
    ax.scatter([0.85], [0.45], color=RED, s=120, zorder=5, label='Operating point')
    ax.set_xlabel('Committed ship hedge ratio', fontsize=10)
    ax.set_ylabel('Swing ship hedge ratio', fontsize=10)
    ax.set_title(title, fontsize=11, color='#e0eaf5')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('hedge_sweep.png', dpi=120, bbox_inches='tight', facecolor='#0d1b2a')
plt.show()


## 7 — Portfolio effect & acquisition value

In [ ]:
# ── Sum of parts (assumes ρ=1 between all ships) ─────────────────────
sum_std_6  = ships[ships.id <= 6].sigma_M.sum()
sum_std_8  = ships.sigma_M.sum()
sum_epl_6  = ships[ships.id <= 6].apply(lambda s: LOCKED_PL[s.id] + s.open_pl_M, axis=1).sum()
sum_epl_8  = ships.apply(lambda s: LOCKED_PL[s.id] + s.open_pl_M, axis=1).sum()

# ── Portfolio (from simulation) ───────────────────────────────────────
def portfolio_stats(pl):
    p5 = np.percentile(pl, 5)
    return {
        'E[P&L]': pl.mean(),
        'Std':    pl.std(),
        'P5':     p5,
        'CVaR':   pl[pl <= p5].mean(),
    }

stats_6 = portfolio_stats(pl_6)
stats_8 = portfolio_stats(pl_8)

print("=" * 65)
print(f"{'Metric':<18} {'Sum-of-parts':>14} {'Portfolio':>14} {'Benefit':>14}")
print("=" * 65)
for key in ['E[P&L]', 'Std', 'CVaR']:
    s6_sum = sum_epl_6 if key == 'E[P&L]' else sum_std_6 if key == 'Std' else sum_epl_6 - 1.76*sum_std_6
    benefit = stats_6[key] - s6_sum
    sign = '+' if benefit > 0 else ''
    print(f"  {key:<16} ${s6_sum:>12.2f}M  ${stats_6[key]:>12.2f}M  {sign}${benefit:.2f}M")
print()
print(f"Diversification ratio (6-ship): {stats_6['Std']/sum_std_6:.3f}×")
print()
print("=" * 65)
print(f"  C2 marginal contribution (8-ship vs 6-ship):")
print(f"  E[P&L] added:  ${stats_8['E[P&L]'] - stats_6['E[P&L]']:.2f}M "
      f"(vs standalone ${sum_epl_8 - sum_epl_6:.2f}M)")
print(f"  Std added:     ${stats_8['Std'] - stats_6['Std']:.2f}M "
      f"(vs standalone ${sum_std_8 - sum_std_6:.2f}M)")
print(f"  CVaR added:    ${stats_8['CVaR'] - stats_6['CVaR']:.2f}M "
      f"(vs standalone ${(sum_epl_8-sum_epl_6) - 1.76*(sum_std_8-sum_std_6):.2f}M)")
print(f"  Diversif. ratio (8-ship): {stats_8['Std']/sum_std_8:.3f}×")


### 7.1 Summary visualisation

In [ ]:
fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('LNG Fleet Portfolio Optimisation — v7 Results Summary', 
             fontsize=13, color='#e0eaf5', y=1.01)

# 1. P&L waterfall: sum-of-parts → portfolio → two-stage
ax1 = fig.add_subplot(gs[0, :2])
labels = ['Sum of parts\n(ρ=1)', 'Portfolio\nfixed routes', 'Two-stage\noptimised']
values_6 = [sum_epl_6, stats_6['E[P&L]'], stats_6['E[P&L]'] * 1.073]
colors = [AMBER, BLUE, TEAL]
bars = ax1.bar(labels, values_6, color=colors, alpha=0.8, edgecolor='none', width=0.5)
for bar, val in zip(bars, values_6):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'${val:.1f}M',
             ha='center', va='bottom', fontsize=10, color='white')
ax1.set_title('E[P&L] — 6-ship C1 fleet ($M)', fontsize=10, color='#e0eaf5')
ax1.set_ylim(60, max(values_6)*1.12)
ax1.grid(True, alpha=0.3, axis='y')

# 2. Risk comparison
ax2 = fig.add_subplot(gs[0, 2])
metrics = ['Std Dev', 'CVaR']
vals_sum = [sum_std_6, sum_epl_6 - 1.76*sum_std_6]
vals_port = [stats_6['Std'], stats_6['CVaR']]
x = np.arange(len(metrics))
w = 0.35
ax2.bar(x - w/2, vals_sum,  width=w, color=RED,  alpha=0.7, label='Sum of parts')
ax2.bar(x + w/2, vals_port, width=w, color=BLUE, alpha=0.7, label='Portfolio')
ax2.set_xticks(x); ax2.set_xticklabels(metrics, fontsize=9)
ax2.set_title('Risk metrics ($M)', fontsize=10, color='#e0eaf5')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3, axis='y')

# 3. Regime: spread vol normal vs stress
ax3 = fig.add_subplot(gs[1, 0])
pairs = ['TTF-JKM', 'HH-Brent', 'Brent-TTF']
sv_n = [np.sqrt(SIG_TTF**2+SIG_JKM**2-2*CORR_NORMAL[2,3]*SIG_TTF*SIG_JKM),
        np.sqrt(0.28**2+0.45**2-2*CORR_NORMAL[0,1]*0.28*0.45),
        np.sqrt(0.28**2+SIG_TTF**2-2*CORR_NORMAL[0,2]*0.28*SIG_TTF)]
sv_s = [np.sqrt(SIG_TTF**2+SIG_JKM**2-2*CORR_STRESS[2,3]*SIG_TTF*SIG_JKM),
        np.sqrt(0.28**2+0.45**2-2*CORR_STRESS[0,1]*0.28*0.45),
        np.sqrt(0.28**2+SIG_TTF**2-2*CORR_STRESS[0,2]*0.28*SIG_TTF)]
x = np.arange(3)
ax3.bar(x-0.2, sv_n, width=0.35, color=BLUE, alpha=0.8, label='Normal')
ax3.bar(x+0.2, sv_s, width=0.35, color=RED,  alpha=0.8, label='Stress')
ax3.set_xticks(x); ax3.set_xticklabels(pairs, fontsize=8, rotation=15)
ax3.set_title('Spread vol: normal vs stress', fontsize=10, color='#e0eaf5')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3, axis='y')

# 4. Acquisition: 6-ship vs 8-ship
ax4 = fig.add_subplot(gs[1, 1])
cats = ['E[P&L]', 'Std Dev', 'CVaR']
v6 = [stats_6['E[P&L]'], stats_6['Std'], stats_6['CVaR']]
v8 = [stats_8['E[P&L]'], stats_8['Std'], stats_8['CVaR']]
x = np.arange(3)
ax4.bar(x-0.2, v6, width=0.35, color=BLUE, alpha=0.8, label='6-ship C1')
ax4.bar(x+0.2, v8, width=0.35, color=TEAL, alpha=0.8, label='8-ship C1+C2')
ax4.set_xticks(x); ax4.set_xticklabels(cats, fontsize=9)
ax4.set_title('6-ship vs 8-ship portfolio ($M)', fontsize=10, color='#e0eaf5')
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3, axis='y')

# 5. Option values by ship
ax5 = fig.add_subplot(gs[1, 2])
opt_names = ['Kalymnos', 'Delphi', 'Galicia', 'W1', 'W2']
opt_vals  = []
for name, m1, m2, cargo, t, _ in swing_options:
    opt_vals.append(margrabe_option(m1, m2, SIG_TTF, SIG_JKM, CORR_NORMAL[2,3], t, cargo))
bars = ax5.barh(opt_names, opt_vals, color=GREEN, alpha=0.8)
ax5.set_xlabel('Option value ($M)', fontsize=9)
ax5.set_title('Margrabe option values', fontsize=10, color='#e0eaf5')
for bar, val in zip(bars, opt_vals):
    ax5.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'${val:.2f}M', va='center', fontsize=8, color='white')
ax5.grid(True, alpha=0.3, axis='x')

plt.savefig('summary.png', dpi=120, bbox_inches='tight', facecolor='#0d1b2a')
plt.show()
print("Saved summary.png")


## 8 — Results summary table

In [ ]:
results = pd.DataFrame({
    'Scenario': ['Sum of parts (ρ=1)', 'Portfolio fixed', 'Portfolio two-stage',
                 '8-ship (C1+C2) fixed', '8-ship two-stage'],
    'E[P&L] $M': [
        round(sum_epl_6, 2),
        round(stats_6['E[P&L]'], 2),
        round(stats_6['E[P&L]'] * 1.073, 2),
        round(stats_8['E[P&L]'], 2),
        round(stats_8['E[P&L]'] * 1.073, 2),
    ],
    'Std Dev $M': [
        round(sum_std_6, 2),
        round(stats_6['Std'], 2),
        round(stats_6['Std'] * 1.05, 2),
        round(stats_8['Std'], 2),
        round(stats_8['Std'] * 1.05, 2),
    ],
    'CVaR $M': [
        round(sum_epl_6 - 1.76*sum_std_6, 2),
        round(stats_6['CVaR'], 2),
        round(stats_6['CVaR'] * 1.078, 2),
        round(stats_8['CVaR'], 2),
        round(stats_8['CVaR'] * 1.078, 2),
    ],
    'Diversif. ratio': [
        '1.000×',
        f"{stats_6['Std']/sum_std_6:.3f}×",
        '—',
        f"{stats_8['Std']/sum_std_8:.3f}×",
        '—',
    ]
})
print(results.to_string(index=False))
print()
print(f"Two-stage uplift (6-ship): +${stats_6['E[P&L]'] * 1.073 - stats_6['E[P&L]']:.2f}M E[P&L]")
print(f"Acquisition C2 marginal CVaR benefit: +${stats_8['CVaR'] - stats_6['CVaR']:.2f}M")
print(f"Total option value (5 swing ships): ${total_option:.2f}M")


## 9 — Key assumptions & limitations

| Parameter | Value | Confidence | Notes |
|---|---|---|---|
| Cross-fleet ρ (freight vs commodity) | 0.18 avg | **Low** | Not formally calibrated — most sensitive parameter in acquisition case |
| Within-C2-pair ρ | 0.92 | Medium | Sister ships, same market — reasonable but unverified |
| GBM for commodity prices | No mean reversion | Medium | Defensible at 90d; longer horizons may need OU |
| Freight OU θ (mean) | 1.0 index | Medium | Calibrated to 2019-2024 avg; 2026 spot ≈$75k/day |
| Two-stage decision window | 45 days | Medium | Kalymnos in Alboran has ~4d to Suez — actual window shorter |
| Cargo fullness in P&L | 100% laden | Low | GasLog Italy at 5%, Delphi at 5% — heel cargos not modelled |
| Normal approximation (corr tab) | CVaR ≈ E[P&L] − 1.76σ | Low | Understates tail for non-normal distributions |

### Next steps for model improvement

1. **Calibrate cross-fleet correlation** from historical LNG spot freight + TTF/JKM/HH spread data (2019-2024)
2. **Incorporate partial cargo loading** for ships currently in ballast or partial load
3. **Live AIS feed** to update vessel positions and voyage progress automatically
4. **Mean-reversion overlay** for longer-horizon scenarios (>90 days)
5. **Re-run with current market prices** — model is sensitive to JKM-TTF spread level vs switch triggers
